
# ¿Qué es un Feature Store?

El **Feature Store** es una herramienta que permite:

- Almacenar, gestionar y compartir características (*features*) utilizadas en modelos de *machine learning*.
- Facilitar la reutilización de datos.
- Asegurar la consistencia entre entrenamiento y producción.
- Mejorar la colaboración entre equipos de ciencia de datos.

## Ventajas principales

- Centraliza el acceso a las features.
- Permite el versionado, trazabilidad y gobernanza de las características.
- Evita la duplicación de esfuerzos.
- Reduce errores en la preparación de datos.
- Permite que los modelos se beneficien de un conjunto de características bien definido y documentado.

## Rol en el flujo de trabajo

En teoría, el Feature Store actúa como un puente entre el procesamiento de datos y el desarrollo de modelos, asegurando que las features sean:

- Accesibles
- Auditables
- Reutilizables

en diferentes proyectos y entornos.

In [0]:
%sql
CREATE TABLE forecasting.default.customer_features (
  customer_id int NOT NULL,
  feat1 long,
  feat2 varchar(100),
  CONSTRAINT customer_features_pk PRIMARY KEY (customer_id)
);

In [0]:
%pip install databricks-feature-engineering

dbutils.library.restartPython()

In [0]:
df_features = spark.table("forecasting.default.customer_features")
display(df_features)

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, IntegerType 

# Definir el esquema para que coincida con la tabla customer_features
schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("feat1", LongType(), True),
    StructField("feat2", StringType(), True)
])

# Ejemplo de nuevas filas a insertar
nuevas_filas = [
    Row(customer_id=1001, feat1=1, feat2="1.2"),
    Row(customer_id=1002, feat1=2, feat2="0.9")
]

# Crear DataFrame de nuevas filas con el esquema correcto
customer_features_df = spark.createDataFrame(nuevas_filas, schema=schema)

display(customer_features_df)

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
fe = FeatureEngineeringClient()
fe.write_table(
  name='forecasting.default.customer_features',
  df = customer_features_df,
  mode = 'merge'
)

In [0]:
df = spark.table("forecasting.default.customer_features")
display(df)